<a href="https://colab.research.google.com/github/abdullahhadi9898/-indata-software-sentiment/blob/main/indata_sentiment_analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#importing data from google drive to collab

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Cell 4 — Verify files are accessible
import os

# Update this path to match your actual folder name in Google Drive
DRIVE_PATH = '/content/drive/MyDrive/indata-project'

# List files in the folder
files = os.listdir(DRIVE_PATH)
print("Files found in your Google Drive folder:")
for f in files:
    size = os.path.getsize(f'{DRIVE_PATH}/{f}') / (1024**3)
    print(f"  {f}  —  {size:.2f} GB")

Files found in your Google Drive folder:
  Software.jsonl.gz  —  0.46 GB
  meta_Software.jsonl.gz  —  0.06 GB


In [1]:
# Cell 5 — Set file paths
REVIEW_FILE = '/content/drive/MyDrive/indata-project/Software.jsonl.gz'
META_FILE = '/content/drive/MyDrive/indata-project/meta_Software.jsonl.gz'
SAMPLE_SIZE = 20000  # 20k per star rating = 100k total

print("✅ File paths set")
print(f"   Review file : {REVIEW_FILE}")
print(f"   Meta file   : {META_FILE}")

✅ File paths set
   Review file : /content/drive/MyDrive/indata-project/Software.jsonl.gz
   Meta file   : /content/drive/MyDrive/indata-project/meta_Software.jsonl.gz


In [5]:
# Cell 6 — Load raw review data
import json
import gzip

print("Loading reviews... this will take a few minutes.")
print("You will see progress updates every 500,000 rows.\n")

reviews = []
with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            reviews.append(json.loads(line.strip()))
        except json.JSONDecodeError:
            continue

        if i % 500000 == 0 and i > 0:
            print(f"  Scanned {i:,} rows so far...")

df_raw = pd.DataFrame(reviews)

print(f"\n✅ Raw data loaded successfully")
print(f"   Total rows    : {df_raw.shape[0]:,}")
print(f"   Total columns : {df_raw.shape[1]}")
print(f"\nColumns found:")
print(df_raw.columns.tolist())

Loading reviews... this will take a few minutes.
You will see progress updates every 500,000 rows.

  Scanned 500,000 rows so far...
  Scanned 1,000,000 rows so far...
  Scanned 1,500,000 rows so far...
  Scanned 2,000,000 rows so far...
  Scanned 2,500,000 rows so far...
  Scanned 3,000,000 rows so far...
  Scanned 3,500,000 rows so far...
  Scanned 4,000,000 rows so far...
  Scanned 4,500,000 rows so far...


NameError: name 'pd' is not defined

importing libraries

In [7]:
# Cell 6+7 (combined) — Install, Import, Load AND Sample in one go
# This avoids memory crashes by never storing all 4.8M rows at once

import subprocess
subprocess.run(['pip', 'install', 'vaderSentiment', 'gensim', 'wordcloud', '-q'])

import json
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')

REVIEW_FILE = '/content/drive/MyDrive/indata-project/Software.jsonl.gz'
META_FILE   = '/content/drive/MyDrive/indata-project/meta_Software.jsonl.gz'
SAMPLE_SIZE = 20000

print("✅ Libraries ready")
print("Loading and sampling simultaneously...")
print("This reads all lines but only keeps 20k per rating.\n")

# Collect reviews grouped by rating as we read
from collections import defaultdict
import random
random.seed(42)

buckets = defaultdict(list)  # one bucket per star rating

with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            record = json.loads(line.strip())
        except json.JSONDecodeError:
            continue

        rating = record.get('rating')
        if rating not in [1.0, 2.0, 3.0, 4.0, 5.0]:
            continue

        # Reservoir sampling — keeps memory flat
        bucket = buckets[rating]
        if len(bucket) < SAMPLE_SIZE:
            bucket.append(record)
        else:
            # Randomly replace an existing item
            j = random.randint(0, i)
            if j < SAMPLE_SIZE:
                bucket[j] = record

        if i % 500000 == 0 and i > 0:
            sizes = {k: len(v) for k, v in buckets.items()}
            print(f"  Scanned {i:,} rows — bucket sizes: {sizes}")

# Combine all buckets into one DataFrame
all_reviews = []
for rating, records in buckets.items():
    all_reviews.extend(records)

df_sample = pd.DataFrame(all_reviews).reset_index(drop=True)

print(f"\n✅ Done")
print(f"   Total rows    : {df_sample.shape[0]:,}")
print(f"   Total columns : {df_sample.shape[1]}")
print(f"\nReviews per star rating:")
print(df_sample['rating'].value_counts().sort_index())

✅ Libraries ready
Loading and sampling simultaneously...
This reads all lines but only keeps 20k per rating.

  Scanned 500,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 1,000,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 1,500,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 2,000,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 2,500,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 3,000,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 3,500,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 4,000,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000, 2.0: 20000, 3.0: 20000}
  Scanned 4,500,000 rows — bucket sizes: {1.0: 20000, 5.0: 20000, 4.0: 20000